# 06. Topic Pool Refinement

`classification_full` 또는 `classification_detail`의 기타 결과를 기반으로 기존 주제 재배치 후보와 신규 주제 후보를 확인합니다.

In [ ]:
import sys
import importlib

from pyspark.sql import functions as F

PROJECT_ROOT = "/Workspace/Users/jungryo.lee@lge.com/prj_TV_voc"
SRC_ROOT = f"{PROJECT_ROOT}/src"

if SRC_ROOT not in sys.path:
    sys.path.append(SRC_ROOT)

import common.config_loader as config_loader
import taxonomy.topic_pool_refiner as topic_pool_refiner

importlib.reload(config_loader)
importlib.reload(topic_pool_refiner)

from common.config_loader import load_config
from taxonomy.topic_pool_refiner import refine_topic_pool_candidates

config = load_config(f"{PROJECT_ROOT}/config/settings.yaml")
print("config loaded")

In [ ]:
# classification_full: 원본 row 기준 운영 분포에서 기타 후보 확인
# classification_detail: memo_id 기준 샘플/분류 검증 결과에서 기타 후보 확인
CLASSIFICATION_INPUT_TABLE_KEY = "classification_full"

MODEL_KEY = "gpt_55"
MODEL_VERSION_VALUE = config["llm"]["models"][MODEL_KEY]["model_version"]
TARGET_CATE_1_DEPTH = "07. 스마트 사용성"
TARGET_CATE_2_DEPTH = "07-01. 채널/컨텐츠 APP"
TARGET_SC_MEASUREMENT = 1

result = refine_topic_pool_candidates(
    spark=spark,
    config=config,
    input_table_key=CLASSIFICATION_INPUT_TABLE_KEY,
    cate_1_depth=TARGET_CATE_1_DEPTH,
    cate_2_depth=TARGET_CATE_2_DEPTH,
    sc_measurement=TARGET_SC_MEASUREMENT,
    model_version=MODEL_VERSION_VALUE,
    prompt_version=config["version"]["prompt_version"],
    taxonomy_version=config["version"]["taxonomy_version"],
)

print({
    "input_table_key": result["input_table_key"],
    "topic_pool_group_count": result["topic_pool_group_count"],
    "detail_cnt": result["detail_df"].count(),
    "others_cnt": result["others_df"].count(),
})
print(result["refinement_config"])

In [ ]:
display(result["group_summary_df"])

In [ ]:
# 기존 topic 후보가 비어 있을 때 원인 확인용
# topic_pool_group_count가 0이면 topic_pool/version/group 매칭 문제입니다.
# match_reason에 기존 topic명이 반복적으로 보이면 refiner 후보 기준 보강 대상입니다.
print({"topic_pool_group_count": result["topic_pool_group_count"]})

display(
    result["others_df"].select(
        "memo",
        "memo_id",
        "pred_topic",
        "classification_stage",
        "match_reason",
    )
    .orderBy(F.rand(42))
    .limit(30)
)

In [ ]:
# 기타로 빠졌지만 기존 topic으로 보낼 가능성이 있는 후보
display(
    result["existing_topic_reassignment_df"]
    .orderBy(F.col("suggestion_score").desc(), F.col("suggested_topic").asc())
    .limit(100)
)

In [ ]:
# 반복되는 기타 패턴 기반 신규 topic 후보
display(
    result["new_topic_candidate_df"].select(
        "cate_1_depth",
        "cate_2_depth",
        "sc_measurement",
        "memo_norm",
        "candidate_cnt",
        "candidate_distinct_memo_id_cnt",
        "candidate_ratio",
        "sample_memo",
        "suggested_action",
        "candidate_reason",
    )
)

In [ ]:
# 기타 원문 샘플 확인
display(
    result["others_df"].select(
        "memo",
        "memo_id",
        "pred_topic",
        "pred_topic_type",
        "classification_stage",
        "match_reason",
    )
    .orderBy(F.rand(42))
    .limit(30)
)